# ICD ReAct v2 Single-Case Notebook

Use this notebook to run one ICD-10-CM case through the v2 manual-navigation agent.

Workflow reflected in the prompt:
1. Identify and search in the Alphabetic Index.
2. Verify and specify in the Tabular List.
3. Reference the Official Guidelines when they can change code choice or sequencing.
4. Aggregate and filter to the final compliant diagnosis code set.

In [10]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path

import mlflow
import mlflow.langchain


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import baselines.icd_react_v2 as icd_react_v2
import baselines.icd_react_v2.agent as icd_react_v2_agent
import baselines.icd_react_v2.single_case as icd_react_v2_single_case

from baselines.icd_react.runtime import resolve_mlflow_experiment, resolve_model_name, resolve_spark_session, resolve_strict_table
from scripts.evaluate_icd_react import load_case_selectors

icd_react_v2_agent = importlib.reload(icd_react_v2_agent)
icd_react_v2_single_case = importlib.reload(icd_react_v2_single_case)
icd_react_v2 = importlib.reload(icd_react_v2)

load_config = icd_react_v2.load_config
fetch_case_record = icd_react_v2.fetch_case_record
build_single_case_prompt = icd_react_v2.build_single_case_prompt
run_single_case_prediction = icd_react_v2.run_single_case_prediction
score_prediction = icd_react_v2.score_prediction
coerce_icd_code_list = icd_react_v2_single_case.coerce_icd_code_list

In [11]:
USE_FIRST_SELECTOR = True
TARGET_HADM_ID = "20389308"
TARGET_SUBJECT_ID = None
TARGET_NOTE_ID = None
MANUAL_SUMMARY = None
MANUAL_EXPECTED_CODES = []
MODEL_NAME = resolve_model_name()
MAX_AGENT_STEPS = 60
STRICT_TABLE = resolve_strict_table()
MLFLOW_EXPERIMENT = resolve_mlflow_experiment()
RUN_MLFLOW_TRACE = True

config = load_config()

print(json.dumps({
    "execution_env": config.execution_env,
    "manuals_root": str(config.manuals_root),
    "strict_table": STRICT_TABLE,
    "model_name": MODEL_NAME,
    "max_agent_steps": MAX_AGENT_STEPS,
    "mlflow_experiment": MLFLOW_EXPERIMENT,
    "use_first_selector": USE_FIRST_SELECTOR,
    "run_mlflow_trace": RUN_MLFLOW_TRACE,
}, indent=2))

{
  "execution_env": "local",
  "manuals_root": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\data\\ICD-2019-manual",
  "strict_table": "usdo_aa_catalog.research_tam_datasets.mimic_icd10_note_dataset_2017_2019_strict",
  "model_name": "openai:gpt-5",
  "max_agent_steps": 60,
  "mlflow_experiment": "/Users/sonutka@mfcgd.com/icd-react-single-case",
  "use_first_selector": true,
  "run_mlflow_trace": true
}


In [12]:
if MANUAL_SUMMARY:
    spark = None
    selector = None
    case_record = {
        "hadm_id": None,
        "subject_id": None,
        "note_id": None,
        "case_summary": str(MANUAL_SUMMARY).strip(),
        "expected_icd_codes": coerce_icd_code_list(MANUAL_EXPECTED_CODES),
    }
else:
    spark = resolve_spark_session(app_name="icd-react-v2-single-case-notebook")
    if USE_FIRST_SELECTOR and TARGET_HADM_ID is None and TARGET_SUBJECT_ID is None and TARGET_NOTE_ID is None:
        selector = load_case_selectors(spark=spark, table_name=STRICT_TABLE, limit=1)[0]
    else:
        selector = {
            "hadm_id": TARGET_HADM_ID,
            "subject_id": TARGET_SUBJECT_ID,
            "note_id": TARGET_NOTE_ID,
        }

    case_record = fetch_case_record(
        spark=spark,
        table_name=STRICT_TABLE,
        hadm_id=int(selector["hadm_id"]) if selector.get("hadm_id") is not None else None,
        subject_id=int(selector["subject_id"]) if selector.get("subject_id") is not None else None,
        note_id=str(selector["note_id"]) if selector.get("note_id") is not None else None,
    )

{
    "hadm_id": case_record.get("hadm_id"),
    "subject_id": case_record.get("subject_id"),
    "note_id": case_record.get("note_id"),
    "expected_icd_codes": case_record.get("expected_icd_codes", []),
    "summary_length": len(case_record.get("case_summary") or ""),
}

{'hadm_id': 20389308,
 'subject_id': 14574546,
 'note_id': '14574546-DS-3',
 'expected_icd_codes': ['M65841', 'B009'],
 'summary_length': 1419}

In [13]:
summary_text = str(case_record.get("case_summary") or "")
print(f"Summary length: {len(summary_text)} characters")
print()
print(summary_text[:4000])

Summary length: 1419 characters

Name:  ___                  Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Service: ORTHOPAEDICS
 
Allergies: 
Keflex
 
Attending: ___.
 
Chief Complaint:
Right long finger suppurative flexor tenosynovitis
 
Major Surgical or Invasive Procedure:
I&D right long finger
 
Brief Hospital Course:
Patient was admitted for operative drainage of suppurative 
flexor tenosynovitis.  Cultures were obtained and patient was 
placed on IV antibiotics.  By POD2, the patient had clinically 
improved.  Therefore, the decision was made to discharge him 
home on oral antibiotics with close follow up as an outpatient.
 
Medications on Admission:
see OMR
 
Discharge Medications:
1.  Acetaminophen 650 mg PO Q6H:PRN pain, HA, T>100 degrees    
2.  cefaDROXil 500 mg oral BID finger infection *AST Approval 
Required*
RX *cefadroxil 500 mg 1 capsule(s) by mouth twice a day Disp 
#*14 Capsule Refills:*0 

 
Di

In [14]:
prompt_text = build_single_case_prompt(summary_text=summary_text)
print(prompt_text[:6000])

You are reviewing one clinical case for ICD-10-CM diagnosis coding.

Follow this workflow exactly:
1. Identify and Search: identify the active diagnoses, symptoms, and clearly active chronic conditions for this encounter, then use the Alphabetic Index tools to find candidate code families.
2. Verify and Specify: inspect the Tabular List for every final code you keep, and only keep the most specific code supported by the case facts and the manual text.
3. Reference Rules: inspect the Official Guidelines when general or chapter-specific rules could change the answer, especially for diabetes, sepsis, symptom-versus-diagnosis coding, combination codes, additional-code requirements, sequencing, or mutually exclusive code choices.
4. Aggregate and Filter: return the final compliant ICD-10-CM diagnosis code set after removing unsupported, redundant, or guideline-blocked codes.

Task:
- Use only the ICD manual tools to navigate the Index, Tabular List, and Guidelines.
- Focus on conditions tha

In [15]:
if mlflow.active_run() is not None:
    mlflow.end_run()

if RUN_MLFLOW_TRACE:
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    mlflow.langchain.autolog(log_traces=True, silent=True)

{
    "tracking_uri": mlflow.get_tracking_uri(),
    "experiment": MLFLOW_EXPERIMENT if RUN_MLFLOW_TRACE else None,
    "run_mlflow_trace": RUN_MLFLOW_TRACE,
}

{'tracking_uri': 'databricks://aa-aab-adb',
 'experiment': '/Users/sonutka@mfcgd.com/icd-react-single-case',
 'run_mlflow_trace': True}

In [16]:
if RUN_MLFLOW_TRACE:
    with mlflow.start_run(run_name=f"icd_react_v2_{case_record.get('hadm_id') or 'manual'}") as run:
        mlflow.log_params({
            "baseline": "icd_react_v2",
            "hadm_id": case_record.get("hadm_id"),
            "subject_id": case_record.get("subject_id"),
            "note_id": case_record.get("note_id"),
            "model_name": MODEL_NAME,
            "max_agent_steps": MAX_AGENT_STEPS,
        })
        result = run_single_case_prediction(
            config=config,
            summary_text=summary_text,
            model_name=MODEL_NAME,
            max_agent_steps=MAX_AGENT_STEPS,
        )
        metrics = score_prediction(
            predicted_codes=result["prediction"].get("predicted_icd_codes", []),
            expected_codes=case_record.get("expected_icd_codes", []),
        )
        mlflow.log_dict(result["prediction"], "single_case/prediction.json")
        mlflow.log_dict({"messages": result["messages"]}, "single_case/messages.json")
        mlflow.log_dict(metrics, "single_case/metrics.json")
        run_id = run.info.run_id
else:
    result = run_single_case_prediction(
        config=config,
        summary_text=summary_text,
        model_name=MODEL_NAME,
        max_agent_steps=MAX_AGENT_STEPS,
    )
    metrics = score_prediction(
        predicted_codes=result["prediction"].get("predicted_icd_codes", []),
        expected_codes=case_record.get("expected_icd_codes", []),
    )
    run_id = None

{
    "mlflow_run_id": run_id,
    "prediction": result["prediction"],
    "metrics": metrics,
}

🏃 View run icd_react_v2_20389308 at: https://adb-2498687920111167.7.azuredatabricks.net/ml/experiments/2273108800187843/runs/c5d5aa5248e543a69c8a1d6ba18faa80
🧪 View experiment at: https://adb-2498687920111167.7.azuredatabricks.net/ml/experiments/2273108800187843


{'mlflow_run_id': 'c5d5aa5248e543a69c8a1d6ba18faa80',
 'prediction': {'predicted_icd_codes': ['M65041', 'Z881'],
  'confidence': 0.9,
  'rationale': 'Suppurative flexor tenosynovitis of the right long finger indicates an abscess of the tendon sheath of the right hand; allergy to Keflex (cephalosporin antibiotic) is a relevant status affecting management.',
  'supporting_evidence': ['Alphabetic Index: Abscess, tendon (sheath), hand -> M65.04-; Tabular M65.041 = Abscess of tendon sheath, right hand (more specific than M65.141 for infective tenosynovitis).',
   'Alphabetic Index: History, personal (of), allergy (to), antibiotic agent -> Z88.1; Tabular Z88.1 = Allergy status to other antibiotic agents.']},
 'metrics': {'predicted_count': 2,
  'expected_count': 2,
  'true_positive_count': 0,
  'precision': 0.0,
  'recall': 0.0}}

In [17]:
{
    "expected_icd_codes": case_record.get("expected_icd_codes", []),
    "predicted_icd_codes": result["prediction"].get("predicted_icd_codes", []),
    "precision": metrics.get("precision"),
    "recall": metrics.get("recall"),
    "supporting_evidence": result["prediction"].get("supporting_evidence", []),
}

{'expected_icd_codes': ['M65841', 'B009'],
 'predicted_icd_codes': ['M65041', 'Z881'],
 'precision': 0.0,
 'recall': 0.0,
 'supporting_evidence': ['Alphabetic Index: Abscess, tendon (sheath), hand -> M65.04-; Tabular M65.041 = Abscess of tendon sheath, right hand (more specific than M65.141 for infective tenosynovitis).',
  'Alphabetic Index: History, personal (of), allergy (to), antibiotic agent -> Z88.1; Tabular Z88.1 = Allergy status to other antibiotic agents.']}

In [18]:
{
    "final_message_text": result["final_message_text"][:6000],
    "message_count": len(result["messages"]),
}

{'final_message_text': '{\n  "predicted_icd_codes": [\n    "M65.041",\n    "Z88.1"\n  ],\n  "confidence": 0.9,\n  "rationale": "Suppurative flexor tenosynovitis of the right long finger indicates an abscess of the tendon sheath of the right hand; allergy to Keflex (cephalosporin antibiotic) is a relevant status affecting management.",\n  "supporting_evidence": [\n    "Alphabetic Index: Abscess, tendon (sheath), hand -> M65.04-; Tabular M65.041 = Abscess of tendon sheath, right hand (more specific than M65.141 for infective tenosynovitis).",\n    "Alphabetic Index: History, personal (of), allergy (to), antibiotic agent -> Z88.1; Tabular Z88.1 = Allergy status to other antibiotic agents."\n  ]\n}',
 'message_count': 32}